# Meter Maintenance Notebook

Use the main notebook for the normal pipeline run.

- inspect one meter or a few meters closely
- compare raw data to the current broken meter source entry (on `running_list_broken_meters.xlsx`)
- check whether a meter is still broken, was repaired, or was not actually broken
- update the source `running_list_broken_meters.xlsx` file in a controlled way
- keep a small update log for review decisions

section ideas:

load one meter or selected meters
plot raw data around chosen dates
compare against current broken list entry
decide update: broken / repaired / not actually broken
save changes back to the source file / or database


In [3]:
import os
import importlib
import numpy as np
import pandas as pd
import data_clean_TEST as dc
importlib.reload(dc)

<module 'data_clean_TEST' from '/Users/cassiehuber/Documents/GitHub/harvest_kwh_prep/notebooks/data_clean_TEST.py'>

### 1. Parameters

In [ ]:
############ CHANGE PARAMETERS AS NEEDED #############

# one meter or a short list of meters to inspect
meters_to_review = [
    # "pbrc_main_b",
]

# optional zoom window for plots
zoom_start = None   # e.g. "2025-08-01 00:00:00"
zoom_end = None     # e.g. "2025-10-01 00:00:00"

# whether to save changes back to the source broken meter file
write_changes_to_source_file = False

######################################################

In [ ]:
# Paths
input_dir = "../data/extracts/"
output_dir = "../data/outputs/"
plot_dir = "../data/outputs/plots/"
maintenance_dir = "../data/outputs/maintenance/"

os.makedirs(output_dir, exist_ok=True)
os.makedirs(plot_dir, exist_ok=True)
os.makedirs(maintenance_dir, exist_ok=True)

# TODO: change to server paths if running on server, make be all of its meter data or a subset
# Main raw meter data for this run
var_file = input_dir + "harvest_interval_kwh_" + "250723-251017.csv"

# TODO: FIX

# Main pipeline source/review files
meter_issues_candidates_file = input_dir + "special_meter_candidates.xlsx"
meter_issues_file = input_dir + "special_meters.xlsx"
broken_meters_file = input_dir + "running_list_broken_meters.xlsx"

# Main pipeline generated files
meter_corrections_file = output_dir + "special_meters_corrections_master_sheet.xlsx"
removed_special_meter_data_file = output_dir + "removed_special_meter_data.csv"

# Maintenance files
broken_meter_update_log_file = maintenance_dir + "broken_meter_update_log.csv"
broken_meter_reviewed_copy_file = maintenance_dir + "running_list_broken_meters_reviewed_copy.xlsx"

In [ ]:
# Load raw interval data
#TODO: change to load from server
raw_df = pd.read_csv(var_file)
raw_df["datetime"] = pd.to_datetime(raw_df["datetime"])
raw_df = raw_df.dropna(subset=["datetime"]).copy()

raw_df.head()

In [ ]:
# Pivot to wide meter dataframe and fill missing timestamps
pivoted_df = raw_df.pivot(index="datetime", columns="meter_name", values="meter_reading").reset_index()
full_df = dc.fill_missing_timestamps(pivoted_df, "15min")
data = full_df.set_index("datetime").sort_index()

print(data.shape)
data.head(2)
